<a href="https://colab.research.google.com/github/Aryan289-coder/my-projects/blob/main/Historical_colorization_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Historical Photo Colorizer
#
# DeOldify does the actual colorization (it's still the best general-purpose
# colorizer around), but its output is "modern plausible color" - it doesn't
# know a photo is from 1943 vs 2020, so everything comes out a bit too clean
# and saturated for old film. This wraps DeOldify with a second pass that
# pushes the result toward how a given period's film/plate stock actually
# rendered color - autochrome pastels for the 1920s, Kodachrome punch for
# WWII, faded Ektachrome magenta for the 70s, etc.
#
# Two ways to get period-accurate color here:
#   1. Pick a built-in period from the dropdown (hand-tuned from how those
#      processes are documented to render color).
#   2. Upload a handful of real photos from the period you're working with
#      and calibrate against those instead - this is the "refine on a
#      historical dataset" option, and it beats the built-in presets when
#      you have actual reference material for your specific archive.

!rm -rf /content/DeOldify
%cd /content
!git clone https://github.com/jantic/DeOldify.git
%cd /content/DeOldify

!sed -i '/torch==/d;/torchvision==/d;/Pillow==/d' requirements.txt
!pip install -q -r requirements.txt
!pip uninstall -y pillow scikit-image
!pip install -q Pillow==10.4.0 scikit-image==0.24.0 opencv-python-headless gradio

import sys
sys.path.insert(0, "/content/DeOldify")

import os
import functools
import tempfile
import cv2
import torch
import numpy as np
import gradio as gr
from PIL import Image
from huggingface_hub import hf_hub_download

from deoldify import device
from deoldify.device_id import DeviceId
from deoldify.visualize import get_image_colorizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    print("No GPU detected - colorization will still run, just slower.")

MODELS_DIR = "/content/DeOldify/models"
os.makedirs(MODELS_DIR, exist_ok=True)


def download_deoldify_model(filename: str) -> None:
    dest = os.path.join(MODELS_DIR, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 100_000_000:
        return
    repo = "databuzzword/deoldify-artistic" if "Artistic" in filename else "databuzzword/deoldify-stable"
    for repo_id in ["thookham/DeOldify", repo]:
        try:
            hf_hub_download(repo_id=repo_id, filename=filename, local_dir=MODELS_DIR)
            return
        except Exception as exc:
            print(f"Failed to download from {repo_id}: {exc}")


download_deoldify_model("ColorizeArtistic_gen.pth")

device.set(device=DeviceId.GPU0 if DEVICE.type == "cuda" else DeviceId.CPU)
_orig_load = torch.load
torch.load = functools.partial(torch.load, weights_only=False)
deoldify_colorizer = get_image_colorizer(artistic=True)
torch.load = _orig_load


def deoldify_colorize(pil_img: Image.Image, render_factor: int) -> np.ndarray:
    pil_img = pil_img.convert("RGB")
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        path = tmp.name
    pil_img.save(path)
    result = deoldify_colorizer.get_transformed_image(path, render_factor=render_factor, watermarked=False)
    os.remove(path)
    return np.array(result.resize(pil_img.size, Image.LANCZOS))


# ---------------------------------------------------------------------------
# Period color grading
#
# Each profile is built from a small set of knobs that map onto documented
# characteristics of the real film/plate process, not arbitrary numbers:
#   saturation   how vivid the color reproduction was
#   warmth/tint  color cast (LAB b = blue<->yellow, LAB a = green<->magenta)
#   contrast     how "punchy" vs flat the tonal range was
#   grain        how visible the film/plate grain is
#   vignette     corner darkening from period lenses
#   mono_bias    how close to monochrome-with-a-tint (early hand-tinted work)
# ---------------------------------------------------------------------------

PERIODS = {
    "1900s-1910s (Sepia / Albumen)": dict(
        saturation=0.15, warmth=22, tint=-6, contrast=1.05, brightness=8,
        grain=7, vignette=0.28, mono_bias=0.82,
    ),
    "1920s (Autochrome)": dict(
        saturation=0.55, warmth=10, tint=10, contrast=0.90, brightness=6,
        grain=11, vignette=0.16, mono_bias=0.0,
    ),
    "1930s-1940s (WWII Kodachrome)": dict(
        saturation=0.85, warmth=8, tint=-5, contrast=1.14, brightness=-3,
        grain=8, vignette=0.10, mono_bias=0.0,
    ),
    "1950s (Kodachrome)": dict(
        saturation=1.05, warmth=5, tint=-2, contrast=1.16, brightness=4,
        grain=5, vignette=0.05, mono_bias=0.0,
    ),
    "1960s-1970s (Faded Ektachrome)": dict(
        saturation=0.70, warmth=16, tint=12, contrast=0.84, brightness=9,
        grain=13, vignette=0.20, mono_bias=0.0,
    ),
}


def _adjust_saturation(rgb, factor):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] *= factor
    hsv[..., 1] = np.clip(hsv[..., 1], 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def _adjust_warmth_tint(rgb, warmth, tint):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab[..., 1] = np.clip(lab[..., 1] + tint, 0, 255)
    lab[..., 2] = np.clip(lab[..., 2] + warmth, 0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)


def _adjust_contrast_brightness(rgb, contrast, brightness):
    out = rgb.astype(np.float32)
    out = (out - 127.5) * contrast + 127.5 + brightness
    return np.clip(out, 0, 255).astype(np.uint8)


def _apply_mono_bias(rgb, bias):
    if bias <= 0:
        return rgb
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    sepia = np.stack([
        np.clip(gray * 1.07, 0, 255),
        np.clip(gray * 0.94, 0, 255),
        np.clip(gray * 0.74, 0, 255),
    ], axis=-1).astype(np.uint8)
    return cv2.addWeighted(sepia, bias, rgb, 1 - bias, 0)


def _add_grain(rgb, amount):
    if amount <= 0:
        return rgb
    noise = np.random.normal(0, amount, rgb.shape[:2])[..., None]
    out = rgb.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)


def _add_vignette(rgb, strength):
    if strength <= 0:
        return rgb
    h, w = rgb.shape[:2]
    y, x = np.ogrid[:h, :w]
    cx, cy = w / 2, h / 2
    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    max_dist = np.sqrt(cx ** 2 + cy ** 2)
    mask = np.clip(1 - strength * (dist / max_dist) ** 2, 0, 1)[..., None]
    return np.clip(rgb.astype(np.float32) * mask, 0, 255).astype(np.uint8)


def apply_period_grade(rgb: np.ndarray, profile: dict, strength: float = 1.0) -> np.ndarray:
    out = _adjust_saturation(rgb, profile["saturation"])
    out = _adjust_warmth_tint(out, profile["warmth"], profile["tint"])
    out = _adjust_contrast_brightness(out, profile["contrast"], profile["brightness"])
    out = _apply_mono_bias(out, profile["mono_bias"])
    out = _add_grain(out, profile["grain"])
    out = _add_vignette(out, profile["vignette"])
    if strength < 1.0:
        out = cv2.addWeighted(out, strength, rgb, 1 - strength, 0)
    return out


def calibrate_period_from_references(image_files, blend=0.6):
    """
    Measures actual saturation/warmth/tint from a set of real reference
    photos and nudges the closest built-in profile toward those measured
    values. This is the "refine against a historical dataset" path - if you
    have real photos from the archive you're colorizing, this will usually
    match that archive's look better than any hand-tuned preset.
    """
    if len(image_files) < 3:
        raise ValueError("Upload at least 3 reference images for a stable estimate.")

    sat_vals, warm_vals, tint_vals = [], [], []
    for f in image_files:
        img = Image.open(f).convert("RGB")
        rgb = np.array(img)
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
        sat_vals.append(hsv[..., 1].mean() / 255 * 2.0)
        warm_vals.append(lab[..., 2].mean() - 128)
        tint_vals.append(lab[..., 1].mean() - 128)

    measured = dict(
        saturation=float(np.mean(sat_vals)),
        warmth=float(np.mean(warm_vals)),
        tint=float(np.mean(tint_vals)),
    )

    closest = min(PERIODS, key=lambda k: abs(PERIODS[k]["saturation"] - measured["saturation"]))
    calibrated = dict(PERIODS[closest])
    for key in ("saturation", "warmth", "tint"):
        calibrated[key] = calibrated[key] * (1 - blend) + measured[key] * blend

    return calibrated, closest


# ---------------------------------------------------------------------------
# Pipeline + UI
# ---------------------------------------------------------------------------

_calibrated_cache = {}  # holds a calibrated profile for this session, if any


def run_calibration(ref_files):
    if not ref_files:
        return "Upload at least 3 reference photos first."
    try:
        profile, closest_base = calibrate_period_from_references(ref_files)
    except ValueError as e:
        return str(e)
    _calibrated_cache["profile"] = profile
    _calibrated_cache["base"] = closest_base
    return (
        f"Calibrated from {len(ref_files)} reference photos "
        f"(closest built-in match: {closest_base}). "
        f"Select 'Custom (calibrated)' below to use it."
    )


def run_pipeline(photo, period_choice, render_factor, grade_strength):
    if photo is None:
        return None, "Please upload an image."

    if period_choice == "Custom (calibrated)":
        if "profile" not in _calibrated_cache:
            return None, "No calibration yet - upload reference photos and click 'Calibrate' first."
        profile = _calibrated_cache["profile"]
        note_period = f"custom profile calibrated from your reference photos"
    else:
        profile = PERIODS[period_choice]
        note_period = period_choice

    colorized = deoldify_colorize(Image.fromarray(photo), render_factor)
    graded = apply_period_grade(colorized, profile, strength=grade_strength)

    note = f"DeOldify colorization graded for: {note_period} (grade strength {grade_strength:.2f})"
    return graded, note


PERIOD_CHOICES = list(PERIODS.keys()) + ["Custom (calibrated)"]

with gr.Blocks(title="Historical Photo Colorizer") as demo:
    gr.Markdown(
        "# Historical Photo Colorizer\n"
        "Colorizes with DeOldify, then grades the result toward how a "
        "specific period's film or plate stock actually rendered color."
    )

    with gr.Row():
        with gr.Column():
            photo_in = gr.Image(label="Upload photo to colorize", type="numpy")
            period_choice = gr.Dropdown(choices=PERIOD_CHOICES, value=PERIOD_CHOICES[0], label="Historical period")
            render_factor = gr.Slider(10, 40, value=30, step=1, label="DeOldify render factor")
            grade_strength = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label="Period grade strength")
            run_btn = gr.Button("Colorize", variant="primary")
        with gr.Column():
            output_img = gr.Image(label="Result")
            status = gr.Textbox(label="Method used", interactive=False)

    run_btn.click(
        fn=run_pipeline,
        inputs=[photo_in, period_choice, render_factor, grade_strength],
        outputs=[output_img, status],
    )

    with gr.Accordion("Optional: calibrate a period from your own reference photos", open=False):
        gr.Markdown(
            "Got real photos from the specific archive/period you're colorizing? "
            "Upload at least 3 and calibrate - this usually beats the built-in "
            "presets because it's tuned to your actual material, not a generic guess."
        )
        ref_files = gr.File(label="Reference photos", file_count="multiple", type="filepath")
        calibrate_btn = gr.Button("Calibrate")
        calibrate_status = gr.Textbox(label="Calibration status", interactive=False)
        calibrate_btn.click(fn=run_calibration, inputs=[ref_files], outputs=[calibrate_status])

if __name__ == "__main__":
    demo.queue().launch(share=True, inline=True, show_error=True)

/content
Cloning into 'DeOldify'...
remote: Enumerating objects: 2620, done.
remote: Counting objects: 100% (583/583), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 2620 (delta 501), reused 459 (delta 459), pack-reused 2037 (from 1)
Receiving objects: 100% (2620/2620), 69.67 MiB | 14.42 MiB/s, done.
Resolving deltas: 100% (1210/1210), done.
/content/DeOldify
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.3/237.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.3/915.3 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB

/content/DeOldify/deoldify/visualize.py:223: SyntaxWarning: invalid escape sequence '\.'
  if re.search('.*?\.jpg', f):
/content/DeOldify/fastai/gen_doc/gen_notebooks.py:63: SyntaxWarning: invalid escape sequence '\s'
  match = re.match(f"^({key})\s*=\s*.*", codestr)
/content/DeOldify/fastai/gen_doc/gen_notebooks.py:216: SyntaxWarning: invalid escape sequence '\('
  if re.search(f"update_nb_metadata\('{fn}'", c['source']): return c
/content/DeOldify/fastai/gen_doc/nbdoc.py:26: SyntaxWarning: invalid escape sequence '\*'
  arg_prefixes = {inspect._VAR_POSITIONAL: '\*', inspect._VAR_KEYWORD:'\*\*'}
/content/DeOldify/fastai/gen_doc/nbdoc.py:26: SyntaxWarning: invalid escape sequence '\*'
  arg_prefixes = {inspect._VAR_POSITIONAL: '\*', inspect._VAR_KEYWORD:'\*\*'}
/content/DeOldify/fastai/gen_doc/nbdoc.py:56: SyntaxWarning: invalid escape sequence '\['
  return f'`Optional`\[{type_repr(args[0])}\]'
/content/DeOldify/fastai/gen_doc/nbdoc.py:56: SyntaxWarning: invalid escape sequence '\]'
 

No GPU detected - colorization will still run, just slower.


INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"


HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"


...

HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"


HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
* Running on public URL: https://1f1ac075b322ac8862.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


INFO:httpx:HTTP Request: HEAD https://1f1ac075b322ac8862.gradio.live "HTTP/1.1 200 OK"


HTTP Request: HEAD https://1f1ac075b322ac8862.gradio.live "HTTP/1.1 200 OK"
